In [ ]:
# Setup

!pip install transformers torch accelerate
!pip install -U "bitsandbytes>=0.46.1"

# 1. Install LLVM and Clang
!sudo apt-get update
!sudo apt-get install -y clang-13 llvm-13 llvm-13-dev llvm-13-tools

# 2. Install the Z3 Theorem Prover
!sudo apt-get install -y z3 libz3-dev

# 3. Install build dependencies
!sudo apt-get install -y build-essential cmake curl git libcap-dev libncurses5-dev python3 python3-pip unzip libtcmalloc-minimal4 libgoogle-perftools-dev libsqlite3-dev

# 4. Clone KLEE and build
!git clone https://github.com/klee/klee.git
!cd klee && mkdir build && cd build && \
cmake -DENABLE_SOLVER_Z3=ON \
      -DENABLE_POSIX_RUNTIME=OFF \
      -DENABLE_KLEE_UCLIBC=OFF \
      -DLLVM_CONFIG_BINARY=/usr/bin/llvm-config-13 .. && \
make -j$(nproc)

!cd klee/build && sudo make install

# Connect to drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code to test Baseline models

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)

test_prompts = [
    "Review this STM32 HAL GPIO initialization snippet. Is it safe to use |= when setting the GPIO_MODER register for an output pin, or does it require a different approach? Provide the corrected C code.",
    "I am writing a FreeRTOS application. Inside an interrupt service routine (ISR), I need to send a message to a queue to wake up a high-priority task. Can I use xQueueSend()? If not, rewrite the ISR correctly.",
    "Identify the MISRA C:2012 violations in this macro: #define SQUARE(x) x * x. Provide the compliant fix and explain the rules violated."
]

for i, prompt in enumerate(test_prompts, 1):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
    response = tokenizer.batch_decode([g[len(i):] for i, g in zip(inputs.input_ids, generated_ids)], skip_special_tokens=True)[0]

    print(f"Test {i} Baseline 7B Response:\n{response}\n")
    print("="*50 + "\n")

In [ ]:
# Code to test Finetuned models

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. Paths
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/ECE285Project/Qwen7B/checkpoint-2000"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load base model with quantization and let accelerate handle the device map
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

# Attach the adapter to the quantized base model
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# 2. The Adversarial Test Suite
test_prompts = [
    "Review this STM32 HAL GPIO initialization snippet. Is it safe to use |= when setting the GPIO_MODER register for an output pin, or does it require a different approach? Provide the corrected C code.",
    "I am writing a FreeRTOS application. Inside an interrupt service routine (ISR), I need to send a message to a queue to wake up a high-priority task. Can I use xQueueSend()? If not, rewrite the ISR correctly.",
    "Identify the MISRA C:2012 violations in this macro: #define SQUARE(x) x * x. Provide the compliant fix and explain the rules violated."
]

system_prompt = "You are an expert embedded firmware auditor specializing in MISRA C:2012 and FreeRTOS safety standards."

for i, prompt in enumerate(test_prompts, 1):
    print(f"Executing Test {i}")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    # Apply the Qwen chat template
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1
        )

        input_length = inputs.input_ids.shape[1]
        response_ids = generated_ids[0][input_length:]
        response = tokenizer.decode(response_ids, skip_special_tokens=True)

    print(f"\nRESULT TEST {i}:\n{response}")
    print("\n" + "="*60 + "\n")

In [ ]:
# Mock header file

mock_header_content = """
#ifndef STM32_MOCK_H
#define STM32_MOCK_H

#include <stdint.h>
#include <klee/klee.h>

/* 1. Define the exact structure of the STM32 GPIO registers */
typedef struct {
  uint32_t MODER;
  uint32_t OTYPER;
  uint32_t OSPEEDR;
  uint32_t PUPDR;
  uint32_t IDR;
  uint32_t ODR;
  uint32_t BSRR;
  uint32_t LCKR;
  uint32_t AFR[2];
} GPIO_TypeDef;

/* 2. Instantiate real variables in RAM to act as our fake hardware */
extern GPIO_TypeDef Mock_GPIOA;
extern GPIO_TypeDef Mock_GPIOB;
extern GPIO_TypeDef Mock_GPIOC;

/* 3. Override the standard STM32 macros */
#define GPIOA ((GPIO_TypeDef *) &Mock_GPIOA)
#define GPIOB ((GPIO_TypeDef *) &Mock_GPIOB)
#define GPIOC ((GPIO_TypeDef *) &Mock_GPIOC)

/* 4. Define common HAL bit definitions */
#define GPIO_PIN_5                 ((uint16_t)0x0020)
#define GPIO_MODE_OUTPUT_PP        0x00000001U
#define GPIO_NOPULL                0x00000000U
#define GPIO_SPEED_FREQ_LOW        0x00000000U

/* 5. Helper function to initialize hardware state for KLEE */
static inline void klee_init_hardware(void) {
    klee_make_symbolic(&Mock_GPIOA, sizeof(Mock_GPIOA), "Mock_GPIOA");
    klee_make_symbolic(&Mock_GPIOB, sizeof(Mock_GPIOB), "Mock_GPIOB");
    klee_make_symbolic(&Mock_GPIOC, sizeof(Mock_GPIOC), "Mock_GPIOC");
}

#endif /* STM32_MOCK_H */
"""

# Write the header file to the local Colab filesystem
with open("stm32_mock.h", "w") as f:
    f.write(mock_header_content)

In [ ]:
# Code to load Finetuned Model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Paths
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/ECE285Project/Qwen7B/checkpoint-2000"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

In [ ]:
# Code for KLEE pipeline

import subprocess
import os
import glob
import re

def extract_c_code(llm_response):
    """Extracts raw C code from the LLM's markdown block."""
    match = re.search(r"```c\n(.*?)\n```", llm_response, re.DOTALL)
    return match.group(1).strip() if match else llm_response.strip()

def run_terminal_command(command):
    """Executes a terminal command and captures the output."""
    try:
        result = subprocess.run(
            command, shell=True, check=True,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        return True, result.stdout
    except subprocess.CalledProcessError as e:
        return False, e.stderr

def run_klee_pipeline(c_code, filename="firmware_test.c"):
    """Compiles to LLVM IR and runs KLEE symbolic execution."""
    # 1. Save the generated code
    with open(filename, "w") as f:
        f.write(c_code)

    # 2. Compile to LLVM Bitcode (Requires Clang)
    compile_cmd = f"clang-13 -emit-llvm -c -g -O0 -I/usr/local/include -I./klee/include -I/content/klee/include {filename} -o firmware_test.bc"

    success, output = run_terminal_command(compile_cmd)

    if not success:
        return False, f"Clang Compilation Error:\n{output}"

    # 3. Execute KLEE
    run_terminal_command("klee firmware_test.bc")

    # 4. Parse KLEE output directory for errors
    klee_dir = "klee-last"
    error_files = glob.glob(f"{klee_dir}/*.err")

    if not error_files:
        return True, "Success: KLEE explored all paths with no memory or logic errors."

    # 5. Extract the first error found to feed back to the LLM
    with open(error_files[0], "r") as f:
        error_trace = f.read()

    return False, f"KLEE Execution Error:\n{error_trace}"

In [ ]:
# Code to test KLEE pipeline and LLM output

MAX_RETRIES = 3

system_prompt = (
    "You are an expert embedded firmware engineer specializing in MISRA C:2012 compliance "
    "and strict hardware safety. You generate complete, compilable C code for testing."
)

# base_prompt = """
# Write a safe, bare-metal STM32 initialization function for GPIOA Pin 5. Configure it for Output Push-Pull, No Pull-up/down, and Low Speed.
# Do NOT use the STM32 HAL library, GPIO_InitTypeDef, or HAL_GPIO_Init(). Manipulate the registers directly.
# Ensure it uses a strict read-modify-write pattern (using &= ~ and |=) to prevent bit-overlap.
# Write a main() function that calls klee_init_hardware() first, and then calls your initialization function.
# You must output a complete, compilable C file that includes the KLEE test harness.
# Follow these exact structural requirements:
# 1. #include "stm32_mock.h"
# 2. Instantiate the global mock hardware variables exactly like this:
#    GPIO_TypeDef Mock_GPIOA;
#    GPIO_TypeDef Mock_GPIOB;
#    GPIO_TypeDef Mock_GPIOC;
# 3. Write your bare-metal GPIO initialization function. You may ONLY use the register names defined in the struct: MODER, OTYPER, OSPEEDR, PUPDR, IDR, ODR, BSRR, LCKR, AFR.

# """

base_prompt = """
Write a bare-metal STM32 function called `Init_Dynamic_Pin` that takes a single `uint8_t pin_number` as an argument.
The function must configure that specific pin on GPIOA to Output Push-Pull mode.
Do NOT use the STM32 HAL library. Manipulate the bare-metal registers directly using a strict read-modify-write pattern.
You must output a complete, compilable C file that includes the KLEE test harness.
Follow these exact structural requirements:
1. #include "stm32_mock.h"
2. Instantiate the global mock hardware variables exactly like this:
   GPIO_TypeDef Mock_GPIOA;
   GPIO_TypeDef Mock_GPIOB;
   GPIO_TypeDef Mock_GPIOC;
3. Write your Init_Dynamic_Pin(uint8_t pin_number) function.
4. Write a main() function that:
   - Calls klee_init_hardware()
   - Declares a uint8_t variable named 'test_pin'
   - Makes 'test_pin' symbolic using klee_make_symbolic(&test_pin, sizeof(test_pin), "test_pin");
   - Calls Init_Dynamic_Pin(test_pin);
"""

current_prompt = base_prompt
success = False

print("\n--- Starting KLEE Verification Loop ---\n")

for attempt in range(1, MAX_RETRIES + 1):
    print(f"--- Attempt {attempt} ---")

    # 1. Prepare ChatML Messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": current_prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 2. Generate Code
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.1,
            repetition_penalty=1.1
        )

    response_ids = generated_ids[0][inputs.input_ids.shape[1]:]
    llm_response = tokenizer.decode(response_ids, skip_special_tokens=True)

    # 3. Extract and verify
    c_code = extract_c_code(llm_response)
    print("Generated Code:\n", c_code, "\n")

    is_safe, feedback = run_klee_pipeline(c_code)

    if is_safe:
        print("\n Verification passed")
        print("Final Safe Code saved to firmware_test.c")
        success = True
        break
    else:
        print(f"\nVERIFICATION FAILED on Attempt {attempt}.")
        print("\n" + "="*40)
        print("EXACT KLEE ERROR TRACE:")
        print("="*40)
        print(feedback)
        print("="*40 + "\n")

        print("Feeding error trace back to model\n")

        # Append the KLEE error to the prompt for the next loop iteration
        current_prompt = f"{base_prompt}\n\nYour previous code failed verification. Here is the KLEE error trace:\n{feedback}\n\nPlease fix the C code."

if not success:
    print("\nMaximum retries reached. Model could not resolve the KLEE errors.")